# 第 12 课 · Scrapy 框架入门

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课只做一件事：理解“为什么抓几百个页面时，手搓 requests 不够用了”，并认识爬虫框架 Scrapy 的分工。
> 只引入三个新概念：**框架解决什么、Scrapy 六组件、Item 与 Pipeline**。

In [1]:
# 每本 notebook 开头都要重新 import
from pathlib import Path
import scrapy                              # 框架本体
from scrapy.http import HtmlResponse        # 用来手动造一个“响应”做离线演示
import sys

DATA_DIR = Path(r'C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\08_爬虫\data')
print('scrapy 版本：', scrapy.__version__)
print('数据目录存在吗：', DATA_DIR.exists())


scrapy 版本： 2.19.0
数据目录存在吗： True


## 12.1 先想一个问题：抓 1 个页面 vs 抓 1000 个页面

前面你用 `requests` + `BeautifulSoup` 手搓爬虫，抓一两个页面很顺手。
但如果你要抓**整站 1000 本书**，手搓脚本会冒出一堆麻烦事：

- 下一个页面的网址谁来排队？（你得自己维护一个“待抓清单”）
- 抓过的网址怎么别再抓第二次？（你得自己记集合去重）
- 失败了怎么自动重试？并发怎么开？（你得自己写）
- 抓下来的数据怎么统一清洗、存数据库？（散在代码各处）

**生活化类比**：`requests` 是你一个人在快递点手工打包，寄 5 个包裹没问题；
寄 5000 个包裹，你需要一条**流水线**：有人分拣地址、有人运输、有人入库。
Scrapy 就是这条流水线，它把上面这些“脏活”都标准化好了。

## 12.2 Scrapy 的六个角色（用“餐厅”来记）

Scrapy 不是一个函数，而是一个**分工明确的小工厂**。把它想象成一家餐厅：

| 组件 | 餐厅里对应谁 | 干什么 |
|------|--------------|--------|
| **Engine** | 店长 | 总指挥，在各角色之间传递单子 |
| **Spider** | 点菜的你（你写的代码） | 决定从哪个网址开始、拿到页面后怎么解析 |
| **Scheduler** | 等餐取号台 | 把待抓的网址排队，还负责别重复 |
| **Downloader** | 后厨 | 真正去把网页下载回来 |
| **Item Pipeline** | 传菜洗碗工 | 把解析出的数据清洗、存进文件/数据库 |
| **Middleware** | 门口迎宾/安检 | 统一改请求头、加代理、处理异常 |

> 你真正要动手写的，主要是两个：**Spider**（怎么解析）和 **Pipeline**（怎么存）。
> 其它四个角色，框架已经替你写好了。

## 12.3 一份请求在厂里怎么跑（数据流）

记这一条主线就够了：

```
你在 Spider 里写一个起始网址
   → Engine 把它交给 Scheduler 排队
   → Scheduler 取出来，交给 Downloader 去下载
   → Downloader 拿回网页（Response）
   → Engine 把 Response 交回你的 Spider.parse()
   → 你的 parse() 里 yield 出两样东西：
        · 新网址 Request   → 送回 Scheduler，继续抓下一页
        · 数据 Item        → 送进 Pipeline 清洗入库
```

**关键点**：`parse()` 每解析出一个“新网址”就丢回队列、解析出一条数据就送去入库。
整站就是这样**自己一圈圈滚起来**的，不用你写 while 循环翻页。

## 12.4 离线演示：把本地页面喂给 Scrapy 的解析器

为了不断网也能玩，我们手动造一个 `HtmlResponse`（假装是从网上下载回来的），
里面装的是本地 `books_index.html`。然后用 Scrapy 自带的选择器取数据。

In [2]:
# 读本地 HTML 的字节，假装是服务器返回的
body = (DATA_DIR / 'books_index.html').read_bytes()

# 造一个 Response 对象：给它一个假网址 + 本地 HTML 内容
response = HtmlResponse(
    url='https://books.toscrape.com/catalogue/page-1.html',
    body=body,
    encoding='utf-8'
)
print('Response 造好了，网址：', response.url)


Response 造好了，网址： https://books.toscrape.com/catalogue/page-1.html


Scrapy 的选择器用 `response.css()`。注意它的两个小语法：
- `::text`：取标签里的文字；
- `::attr(href)`：取标签的属性。

In [3]:
# 取所有书的标题文字
titles = response.css('article.product_pod h3 a::text').getall()
# 取所有书的详情链接
hrefs = response.css('article.product_pod h3 a::attr(href)').getall()
# 取所有价格
prices = response.css('article.product_pod p.price_color::text').getall()

print('抓到', len(titles), '本书：')
for t, h, p in zip(titles, hrefs, prices):
    print(f'  {t.strip():28s} | {p.strip():8s} | {h}')


抓到 4 本书：
  A Light in the Attic         | ¥51.77   | catalogue/a-light-in-the-attic_1000/index.html
  Tipping the Velvet           | ¥53.74   | catalogue/tipping-the-velvet_999/index.html
  Soumission                   | ¥50.10   | catalogue/soumission_998/index.html
  Sharp Objects                | ¥47.82   | catalogue/sharp-objects_997/index.html


## 12.5 Item：给数据定一个“表格表头”

**是什么**：`Item` 是一个轻量的数据模板，声明“我要抓哪几个字段”。
**为什么需要**：不写 Item，你随手塞 dict，字段名容易乱（一会儿 `price` 一会儿 `Price`）。
写了 Item，全队（包括 Pipeline）都知道每条数据长什么样。

In [4]:
class BookItem(scrapy.Item):
    title = scrapy.Field()      # 书名
    price = scrapy.Field()       # 价格
    star = scrapy.Field()        # 星级

# 造一条数据看看
item = BookItem(title='示例书', price=51.77, star=3)
print('Item 像字典：', dict(item))
print('它声明的字段：', list(item.fields.keys()))


Item 像字典： {'title': '示例书', 'price': 51.77, 'star': 3}
它声明的字段： ['title', 'price', 'star']


## 12.6 Pipeline：数据入库的“流水线工位”

**是什么**：Pipeline 是一个类，每条抓下来的 Item 都会经过它的 `process_item` 方法。
**为什么需要**：把“存数据”这件事集中在一个地方，和“怎么解析”分开——改存储逻辑不用动 Spider。

下面写一个简单 Pipeline，把每条 Item 打印出来（真实项目里这里会写 CSV / 数据库）：

In [5]:
class PrintPipeline:
    def open_spider(self, spider):
        print('--- 流水线开工 ---')

    def process_item(self, item, spider):
        # 真实项目里：这里校验数据、清洗、写数据库
        print('入库一条：', dict(item))
        return item            # 必须 return item，传给下一个工位

    def close_spider(self, spider):
        print('--- 流水线收工 ---')

# 手动把刚才解析的东西喂给 Pipeline 跑一遍
pipe = PrintPipeline()
pipe.open_spider(spider=None)
for t, p in zip(titles, prices):
    pipe.process_item(BookItem(title=t.strip(), price=float(p.strip().replace('¥','')), star=3), spider=None)
pipe.close_spider(spider=None)


--- 流水线开工 ---
入库一条： {'title': 'A Light in the Attic', 'price': 51.77, 'star': 3}
入库一条： {'title': 'Tipping the Velvet', 'price': 53.74, 'star': 3}
入库一条： {'title': 'Soumission', 'price': 50.1, 'star': 3}
入库一条： {'title': 'Sharp Objects', 'price': 47.82, 'star': 3}
--- 流水线收工 ---


## 12.7 一个完整 Scrapy 工程长什么样（照着看，不用建）

真实项目里你会用命令生成一堆文件。下面把核心 4 个文件贴出来，逐行注释，知道每个文件是干嘛的就行：

**`items.py`**（定义数据模板）
```python
import scrapy

class BookItem(scrapy.Item):
    title = scrapy.Field()   # 书名
    price = scrapy.Field()   # 价格
```

**`spiders/book_spider.py`**（你写的：从哪抓、怎么解析）
```python
import scrapy
from myspider.items import BookItem

class BookSpider(scrapy.Spider):
    name = 'book_spider'                 # 这个爬虫的名字
    start_urls = ['https://books.toscrape.com/catalogue/page-1.html']  # 起点

    def parse(self, response):
        # 每解析出一本书，就 yield 一个 Item（送去 Pipeline）
        for pod in response.css('article.product_pod'):
            yield BookItem(
                title=pod.css('h3 a::text').get().strip(),
                price=pod.css('p.price_color::text').get(),
            )
        # 再找到“下一页”链接，yield 一个新请求（送回队列，继续翻页）
        next_page = response.css('li.next a::attr(href)').get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)
```

**`pipelines.py`**（你写的：怎么存数据）
```python
import json

class JsonLinesPipeline:
    def open_spider(self, spider):
        self.file = open('books.jsonl', 'w', encoding='utf-8')
    def process_item(self, item, spider):
        self.file.write(json.dumps(dict(item), ensure_ascii=False) + '\n')
        return item
    def close_spider(self, spider):
        self.file.close()
```

**`settings.py`**（配置：性能与合规）
```python
ROBOTSTXT_OBEY = True        # 自动遵守 robots.txt（下一课讲）
DOWNLOAD_DELAY = 1.0         # 每次请求间隔 1 秒，礼貌抓取
CONCURRENT_REQUESTS = 8      # 并发数，别盲目调太大
ITEM_PIPELINES = {'myspider.pipelines.JsonLinesPipeline': 300}  # 注册上面的 Pipeline
```

## 12.8 小结（记住这三句）

1. 手搓 requests 抓 1 页很爽，抓 1000 页要自己管队列/去重/重试/入库——Scrapy 把这些标准化了。
2. 你主要写两个东西：**Spider**（怎么解析、怎么翻页）和 **Pipeline**（怎么存）。
3. `parse()` 里 `yield` 数据给 Pipeline，`yield` 新网址回队列——整站就这样自己滚起来。

> 下一课是整个模块最重要的一课：**反爬与合规红线**。技术学会了，更要知道边界在哪。

## 12.9 随堂小练习（自己动手，空格里写）

1. 用上面离线造好的 `response`，取出所有书的**星级**（提示：在 `p.star-rating` 的 class 属性里，第二个词才是星级）。
2. 说一说：如果 `parse()` 里只 `yield` Item、从不 `yield` 下一页请求，会发生什么？

In [6]:
# TODO：自己写代码完成上面两题（随堂小练，没有单独答案文件）


## 来源与延伸

- Scrapy 官方文档（长期稳定核心）：https://docs.scrapy.org
- Scrapy 官方仓库（约 64.7k star，截至 2026-10 调研）：https://github.com/scrapy/scrapy
- 官方练习站：https://books.toscrape.com/
- 崔庆才 Python3WebSpider（中文 Scrapy 实战）：https://github.com/Python3WebSpider
